In [1]:
from predictor import StockPredictor

sp = StockPredictor(ticker="AAPL")
sp.load_data("AAPL.csv")
sp.train()                   # ~60s
results = sp.predict()
StockPredictor.print_summary(results, "AAPL")

Importing plotly failed. Interactive plots will not work.


[AAPL] Loaded 11,450 rows  (1980-12-12 → 2026-05-19)

  Training pipeline for AAPL

[1/4] Engineering features…
      10,946 usable rows, 85 columns (76 feature cols)

[2/4] Training LightGBM (direct multi-horizon)…
  [LGBM] Training horizon=1w (5d)  n_samples=10946
           CV MAPE = 0.0419
  [LGBM] Training horizon=1m (21d)  n_samples=10946
           CV MAPE = 0.0932
  [LGBM] Training horizon=1y (252d)  n_samples=10946
           CV MAPE = 0.3446

[3/4] Fitting Prophet (trend + seasonality)…
  [Prophet] Fitting on 1256 rows (last 5 yrs)…


12:23:50 - cmdstanpy - INFO - Chain [1] start processing
12:23:52 - cmdstanpy - INFO - Chain [1] done processing


  [Prophet] Done. Forecast horizon: 395 cal-days.

[4/4] Loading Chronos pre-trained foundation model…
  [Chronos] Loading amazon/chronos-t5-small on cpu …


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

  [Chronos] Model loaded ✓

✓ Training complete in 76.7s

                   AAPL Forecast Summary                    
  Current price  : $298.97

  1W      ▼ $   291.91  (-2.4%)   CI: [$274.65, $309.13]
          ↳ lgbm     $   293.41  (w=0.79)
          ↳ prophet  $   286.46  (w=0.21)
          LightGBM CV MAPE: 4.19%

  1M      ▼ $   282.97  (-5.4%)   CI: [$272.36, $305.64]
          ↳ lgbm     $   281.72  (w=0.79)
          ↳ prophet  $   287.62  (w=0.21)
          LightGBM CV MAPE: 9.32%

  1Y      ▲ $   360.69  (+20.6%)   CI: [$43.23, $2818.58]
          ↳ lgbm     $   310.87  (w=0.41)
          ↳ prophet  $   400.24  (w=0.59)
          LightGBM CV MAPE: 34.46%



In [ ]:
import joblib

# 1. Clear the Chronos pipeline from memory before saving (it's zero-shot anyway)
if sp.chronos and sp.chronos.pipeline:
    sp.chronos.pipeline = None 

# 2. Save the entire predictor object
joblib.dump(sp, "aapl_stock_predictor.pkl")
print("Pipeline saved successfully!")

# --- TO LOAD IT LATER ---
# loaded_sp = joblib.load("aapl_stock_predictor.pkl")
# loaded_sp.chronos.load() # Re-initialize the Chronos pipeline